# Citi Bike NYC — Dimensión: Utilización histórica de estaciones (U1, batch)

**Integrante:** Grimaldo Arredondo Martinez — Data Geniuses
**Proyecto Sello:** CitiBike Intelligent Mobility
**Arquitectura del equipo:** Lambda (capa batch con PySpark + capa streaming con Kafka/Structured Streaming en Unidad II)
**Alcance de este notebook:** ruta batch (U1) completa — Bronze → Silver → Gold + modelo de clasificación.

## 1. Arquitectura Big Data y justificación (S1)

El equipo eligió **Lambda** porque el caso de negocio necesita dos cosas que una arquitectura Kappa
pura no resuelve igual de bien: (a) entrenar un modelo sobre **todo el histórico** disponible sin
tener que reproducirlo como un flujo, y (b) mantener un pronóstico en vivo (U2, Unidad II) sin
reprocesar el histórico cada vez. La capa **batch** (esta unidad) construye el Data Lake medallón
(Bronze/Silver/Gold) y entrena el primer modelo; la capa **velocidad** (Unidad II) consumirá
`citibike-rides` desde Kafka y usará este mismo histórico solo para entrenar el modelo de streaming.

Para mi dimensión específica (**utilización histórica de estaciones**), la arquitectura Lambda
significa: el ranking y la probabilidad de alta demanda por estación se calculan aquí, en batch,
sobre el histórico completo; en Unidad II esa misma pregunta se responde en vivo, ventana a ventana,
sobre el flujo Kafka — ambas convergen en la misma pregunta central del equipo.

## 2. Pregunta y variable objetivo de esta dimensión

> ¿Qué estaciones concentran históricamente la mayor demanda y cuáles presentan mayor probabilidad de
> registrar niveles elevados de utilización?

* **Indicadores:** viajes iniciados y finalizados por estación, participación y ranking por estación.
* **Variable objetivo del modelo (`es_alta_demanda`):** etiqueta binaria por estación-día que indica si
  ese día la estación estuvo en el **cuartil superior** de viajes totales, con el umbral calculado
  **solo sobre el período de entrenamiento** (criterio estadístico reproducible, sin fuga de información).
* El modelo **no usa el propio volumen de viajes como predictor** (sería circular); predice la
  probabilidad de un día de alta demanda a partir del **patrón** de uso de la estación ese día
  (día de la semana, fin de semana, mezcla de tipo de usuario y tipo de bicicleta) — información que
  se conoce o se puede proyectar con antelación para planificar operación.

## 3. Inicialización de Spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("U1_Estaciones_CitiBike_GrimaldoArredondo")
    .config("spark.driver.memory", "5g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "America/New_York")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)
print("Nucleos disponibles:", spark.sparkContext.defaultParallelism)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/11 19:11:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/09/11 19:11:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.2.0
Nucleos disponibles: 8


## 4. Capa Bronze — Ingesta con esquema explícito (S2)

`start_station_id` y `end_station_id` traen valores como `6364.10`: **parecen** decimales pero son
identificadores de estación. Si se deja que Spark infiera el tipo, los castea a `double` y
`6364.10` se convierte en `6364.1`, corrompiendo silenciosamente cualquier agrupación por estación.
Por eso el esquema se declara explícitamente y ambos IDs se tipan como `StringType`.

In [2]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType,
)

esquema_viajes = StructType([
    StructField("ride_id",            StringType(),    True),
    StructField("rideable_type",      StringType(),    True),
    StructField("started_at",         TimestampType(), True),
    StructField("ended_at",           TimestampType(), True),
    StructField("start_station_name", StringType(),    True),
    StructField("start_station_id",   StringType(),    True),
    StructField("end_station_name",   StringType(),    True),
    StructField("end_station_id",     StringType(),    True),
    StructField("start_lat",          DoubleType(),    True),
    StructField("start_lng",          DoubleType(),    True),
    StructField("end_lat",            DoubleType(),    True),
    StructField("end_lng",            DoubleType(),    True),
    StructField("member_casual",      StringType(),    True),
])

DATA_PATH = "/opt/UNIDAD1/data/*.csv"

df_bronze = (
    spark.read
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss.SSS")
    .schema(esquema_viajes)
    .csv(DATA_PATH)
)

total_bronze = df_bronze.count()
print(f"Registros ingeridos (Bronze): {total_bronze:,}")
df_bronze.printSchema()

Registros ingeridos (Bronze): 4,993,137
root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)



## 5. Transformaciones distribuidas y evaluación perezosa (S2)

Se agregan columnas derivadas con `withColumn()` (temporales, sin acción todavía), se aplica un
`filter()` y se inspecciona el **plan físico** con `explain()` para mostrar en qué momento Spark deja
de ser perezoso: las transformaciones solo construyen el DAG; la acción (`count()`) es lo que dispara
la lectura y el cómputo real sobre el clúster.

In [3]:
from pyspark.sql.functions import col, to_date, hour, dayofweek, when, unix_timestamp

df_feat = (
    df_bronze
    .withColumn("duration_minutes",
                (unix_timestamp(col("ended_at")) - unix_timestamp(col("started_at"))) / 60.0)
    .withColumn("trip_date", to_date(col("started_at")))
    .withColumn("start_hour", hour(col("started_at")))
    .withColumn("day_of_week", dayofweek(col("started_at")))
    .withColumn("is_weekend", when(col("day_of_week").isin(1, 7), 1).otherwise(0))
)

df_filtrado = df_feat.filter(col("start_station_id").isNotNull()).cache()

print("Transformaciones declaradas -- ningun computo ha ocurrido todavia.\n")
df_filtrado.explain(mode="simple")

print("\nAccion (count): aqui si se ejecuta el DAG ->", f"{df_filtrado.count():,}", "filas")
print("(el resultado queda en cache: las siguientes secciones no vuelven a leer los 5 CSV crudos)")

Transformaciones declaradas -- ningun computo ha ocurrido todavia.

== Physical Plan ==
InMemoryTableScan [ride_id#0, rideable_type#1, started_at#2, ended_at#3, start_station_name#4, start_station_id#5, end_station_name#6, end_station_id#7, start_lat#8, start_lng#9, end_lat#10, end_lng#11, member_casual#12, duration_minutes#31, trip_date#32, start_hour#33, day_of_week#34, is_weekend#35]
   +- InMemoryRelation [ride_id#0, rideable_type#1, started_at#2, ended_at#3, start_station_name#4, start_station_id#5, end_station_name#6, end_station_id#7, start_lat#8, start_lng#9, end_lat#10, end_lng#11, member_casual#12, duration_minutes#31, trip_date#32, start_hour#33, day_of_week#34, is_weekend#35], StorageLevel(disk, memory, deserialized, 1 replicas)
         +- *(1) Project [ride_id#0, rideable_type#1, started_at#2, ended_at#3, start_station_name#4, start_station_id#5, end_station_name#6, end_station_id#7, start_lat#8, start_lng#9, end_lat#10, end_lng#11, member_casual#12, duration_minutes#31, 


Accion (count): aqui si se ejecuta el DAG -> 4,990,109 filas
(el resultado queda en cache: las siguientes secciones no vuelven a leer los 5 CSV crudos)


## 6. Agregaciones distribuidas: `groupBy().agg()` (S2)

Construimos dos vistas agregadas: viajes **iniciados** por estación y viajes **finalizados** por
estación, cada una con su desglose por tipo de usuario y tipo de bicicleta.

In [4]:
from pyspark.sql.functions import count as _count

viajes_iniciados = (
    df_filtrado.groupBy(
        col("start_station_id").alias("station_id"),
        col("start_station_name").alias("station_name"),
    )
    .agg(_count("*").alias("viajes_iniciados"))
)

viajes_finalizados = (
    df_filtrado.filter(col("end_station_id").isNotNull())
    .groupBy(
        col("end_station_id").alias("station_id"),
        col("end_station_name").alias("station_name"),
    )
    .agg(_count("*").alias("viajes_finalizados"))
)

print("Estaciones con viajes iniciados:", viajes_iniciados.count())
print("Estaciones con viajes finalizados:", viajes_finalizados.count())
viajes_iniciados.orderBy(col("viajes_iniciados").desc()).show(5, truncate=False)

Estaciones con viajes iniciados: 2305


Estaciones con viajes finalizados: 2419


+----------+------------------------+----------------+
|station_id|station_name            |viajes_iniciados|
+----------+------------------------+----------------+
|6233.04   |Pier 61 at Chelsea Piers|18260           |
|6140.05   |W 21 St & 6 Ave         |16729           |
|5712.03   |Cooper Square & Astor Pl|15730           |
|6492.08   |9 Ave & W 33 St         |14861           |
|6331.01   |W 31 St & 7 Ave         |14767           |
+----------+------------------------+----------------+
only showing top 5 rows


## 7. Procesamiento con la API de RDD (S2)

Para evidenciar el procesamiento a bajo nivel (RDD), replicamos el conteo de viajes iniciados por
estación usando `map` + `reduceByKey` directamente sobre el RDD subyacente del DataFrame, y
verificamos que el resultado coincide con el obtenido por `groupBy().agg()` en la sección anterior.

Se proyecta primero una sola columna (`select("start_station_id")`) antes de pasar a RDD: convertir
Rows completos a objetos Python y serializarlos hacia los procesos worker de PySpark es costoso en
memoria: limitar la columna reduce ese costo drásticamente sobre ~5 millones de registros.

In [5]:
rdd_conteo = (
    df_filtrado.select("start_station_id").rdd
    .map(lambda row: (row[0], 1))
    .reduceByKey(lambda a, b: a + b)
)

print("Particiones del RDD:", rdd_conteo.getNumPartitions())
top5_rdd = rdd_conteo.takeOrdered(5, key=lambda kv: -kv[1])
print("Top 5 estaciones por viajes iniciados (via RDD):")
for station_id, total in top5_rdd:
    print(f"  {station_id}: {total:,}")

total_rdd = rdd_conteo.map(lambda kv: kv[1]).sum()
total_df = viajes_iniciados.agg({"viajes_iniciados": "sum"}).collect()[0][0]
print(f"\nTotal via RDD: {total_rdd:,} | Total via DataFrame: {total_df:,} | Coinciden: {total_rdd == total_df}")

Particiones del RDD: 10


Top 5 estaciones por viajes iniciados (via RDD):
  6233.04: 18,260
  6140.05: 16,729
  5712.03: 15,730
  6492.08: 14,861
  6331.01: 14,767



Total via RDD: 4,990,109 | Total via DataFrame: 4,990,109 | Coinciden: True


## 8. Calidad de datos (S3)

### 8.1 Duplicados

Verificamos duplicados por `ride_id` (un mismo viaje no debería repetirse; los 5 CSV mensuales
podrían solaparse en los bordes de fecha) y los eliminamos documentando el criterio: se conserva una
única fila por `ride_id`.

In [6]:
n_antes = df_filtrado.count()
df_dedup = df_filtrado.dropDuplicates(["ride_id"])
n_despues = df_dedup.count()

print(f"Filas antes de deduplicar : {n_antes:,}")
print(f"Filas despues de dedup    : {n_despues:,}")
print(f"Duplicados eliminados     : {n_antes - n_despues:,}")

Filas antes de deduplicar : 4,990,109
Filas despues de dedup    : 4,990,109
Duplicados eliminados     : 0


### 8.2 Nulos

Para esta dimensión (estaciones), un viaje sin estación de origen o destino no aporta información
utilizable: se descartan explícitamente (decisión documentada), en vez de dejarlos como nulos
silenciosos en las agregaciones posteriores.

In [7]:
from pyspark.sql.functions import count as _count2

nulos_antes = df_dedup.select(
    _count2(when(col("start_station_id").isNull(), 1)).alias("nulos_start_station"),
    _count2(when(col("end_station_id").isNull(), 1)).alias("nulos_end_station"),
    _count2(when(col("start_station_name").isNull(), 1)).alias("nulos_start_name"),
    _count2(when(col("end_station_name").isNull(), 1)).alias("nulos_end_name"),
)
nulos_antes.show()

df_silver = df_dedup.na.drop(
    subset=["start_station_id", "end_station_id", "start_station_name", "end_station_name"]
).cache()

print(f"Filas Silver (sin nulos de estacion): {df_silver.count():,}")
print("(df_silver queda en cache: el ranking, el dataset estacion-dia y el modelo lo reutilizan)")

# df_filtrado ya cumplio su proposito (agregaciones, RDD, deduplicacion y este ultimo
# na.drop); se libera la cache para dejar memoria disponible a Gold y ML.
df_filtrado.unpersist()

+-------------------+-----------------+----------------+--------------+
|nulos_start_station|nulos_end_station|nulos_start_name|nulos_end_name|
+-------------------+-----------------+----------------+--------------+
|                  0|            14702|               0|         13727|
+-------------------+-----------------+----------------+--------------+



Filas Silver (sin nulos de estacion): 4,975,407
(df_silver queda en cache: el ranking, el dataset estacion-dia y el modelo lo reutilizan)


DataFrame[ride_id: string, rideable_type: string, started_at: timestamp, ended_at: timestamp, start_station_name: string, start_station_id: string, end_station_name: string, end_station_id: string, start_lat: double, start_lng: double, end_lat: double, end_lng: double, member_casual: string, duration_minutes: double, trip_date: date, start_hour: int, day_of_week: int, is_weekend: int]

## 9. Construcción del dataset estación-día y del ranking (S3)

Se agregan los viajes por **estación y por día** (granularidad necesaria para el modelo de
clasificación) y, por separado, un **ranking global por estación** (salida descriptiva del notebook).

In [8]:
from pyspark.sql.functions import sum as _sum, avg as _avg, round as _round

df_estacion_dia = (
    df_silver.groupBy(
        col("start_station_id").alias("station_id"),
        col("start_station_name").alias("station_name"),
        "trip_date", "day_of_week", "is_weekend",
    )
    .agg(
        _count("*").alias("viajes_totales_dia"),
        _round(_avg(when(col("member_casual") == "member", 1).otherwise(0)), 4).alias("member_ratio"),
        _round(_avg(when(col("rideable_type") == "electric_bike", 1).otherwise(0)), 4).alias("electric_ratio"),
    )
)

print(f"Filas estacion-dia: {df_estacion_dia.count():,}")
df_estacion_dia.orderBy(col("viajes_totales_dia").desc()).show(5, truncate=False)

Filas estacion-dia: 70,388


+----------+------------------------+----------+-----------+----------+------------------+------------+--------------+
|station_id|station_name            |trip_date |day_of_week|is_weekend|viajes_totales_dia|member_ratio|electric_ratio|
+----------+------------------------+----------+-----------+----------+------------------+------------+--------------+
|6233.04   |Pier 61 at Chelsea Piers|2026-07-22|4          |0         |810               |0.7778      |0.6827        |
|6233.04   |Pier 61 at Chelsea Piers|2026-07-08|4          |0         |786               |0.8028      |0.6514        |
|6233.04   |Pier 61 at Chelsea Piers|2026-07-23|5          |0         |782               |0.7801      |0.6471        |
|6876.04   |Central Park S & 6 Ave  |2026-07-25|7          |1         |781               |0.3227      |0.7324        |
|6876.04   |Central Park S & 6 Ave  |2026-07-12|1          |1         |766               |0.3251      |0.6606        |
+----------+------------------------+----------+

In [9]:
df_ranking = (
    df_silver.groupBy(
        col("start_station_id").alias("station_id"),
        col("start_station_name").alias("station_name"),
    )
    .agg(_count("*").alias("total_viajes_iniciados"))
)

total_global = df_ranking.agg(_sum("total_viajes_iniciados")).collect()[0][0]
df_ranking = (
    df_ranking
    .withColumn("participacion_pct", _round(col("total_viajes_iniciados") / total_global * 100, 4))
    .orderBy(col("total_viajes_iniciados").desc())
)

print("Ranking de estaciones (top 10):")
df_ranking.show(10, truncate=False)

Ranking de estaciones (top 10):


+----------+--------------------------+----------------------+-----------------+
|station_id|station_name              |total_viajes_iniciados|participacion_pct|
+----------+--------------------------+----------------------+-----------------+
|6233.04   |Pier 61 at Chelsea Piers  |18220                 |0.3662           |
|6140.05   |W 21 St & 6 Ave           |16697                 |0.3356           |
|5712.03   |Cooper Square & Astor Pl  |15695                 |0.3155           |
|6492.08   |9 Ave & W 33 St           |14824                 |0.2979           |
|6331.01   |W 31 St & 7 Ave           |14721                 |0.2959           |
|5905.12   |Broadway & E 14 St        |14436                 |0.2901           |
|6876.04   |Central Park S & 6 Ave    |14179                 |0.285            |
|5980.10   |E 17 St & Broadway        |13982                 |0.281            |
|6912.01   |7 Ave & Central Park South|13979                 |0.281            |
|6948.10   |Broadway & W 58 

## 10. Etiqueta de clasificación: `es_alta_demanda` (S3/S4)

El umbral se calcula con `percentile_approx` (percentil 75 de `viajes_totales_dia`) **usando
únicamente las fechas de entrenamiento** (los primeros 24 días del rango disponible), y se aplica de
forma idéntica sobre entrenamiento y prueba — así el criterio es estadístico y reproducible, y no hay
fuga de información desde el conjunto de prueba hacia la definición de la etiqueta.

In [10]:
from pyspark.sql.functions import expr, lit, min as _min, max as _max

rango = df_estacion_dia.agg(_min("trip_date").alias("desde"), _max("trip_date").alias("hasta")).collect()[0]
desde, hasta = rango["desde"], rango["hasta"]
dias_totales = (hasta - desde).days + 1
corte = desde + __import__("datetime").timedelta(days=int(dias_totales * 0.75))

print(f"Rango de fechas: {desde} -> {hasta} ({dias_totales} dias)")
print(f"Corte train/test: {corte}")

df_train_raw = df_estacion_dia.filter(col("trip_date") < lit(corte))
df_test_raw = df_estacion_dia.filter(col("trip_date") >= lit(corte))

umbral_p75 = df_train_raw.select(
    expr("percentile_approx(viajes_totales_dia, 0.75)").alias("p75")
).collect()[0]["p75"]

print(f"Umbral de alta demanda (P75 en train): {umbral_p75} viajes/dia")

df_etiquetado = df_estacion_dia.withColumn(
    "es_alta_demanda", when(col("viajes_totales_dia") >= lit(umbral_p75), 1).otherwise(0)
).cache()

print("\nBalance de clases:")
df_etiquetado.groupBy("es_alta_demanda").count().show()
print("(df_etiquetado queda en cache: lo reutilizan la escritura Gold y el split train/test del modelo)")

Rango de fechas: 2026-06-30 -> 2026-07-31 (32 dias)
Corte train/test: 2026-07-24


Umbral de alta demanda (P75 en train): 93 viajes/dia

Balance de clases:


+---------------+-----+
|es_alta_demanda|count|
+---------------+-----+
|              1|18362|
|              0|52026|
+---------------+-----+

(df_etiquetado queda en cache: lo reutilizan la escritura Gold y el split train/test del modelo)


## 11. Persistencia Gold particionada y verificación de ida y vuelta (S3)

Se escribe en **Parquet**, particionado por `es_alta_demanda` (columna binaria de baja cardinalidad
y usada en filtros de negocio: analistas de operación filtran directamente las estaciones-día de alta
demanda). Se reconcilia el conteo tras leer de vuelta y se confirma `PartitionFilters` en el plan
físico.

In [11]:
GOLD_PATH = "/opt/UNIDAD1/gold/estaciones_utilizacion"

(
    df_etiquetado.write
    .mode("overwrite")
    .partitionBy("es_alta_demanda")
    .parquet(GOLD_PATH)
)
print(f"Capa Gold escrita en: {GOLD_PATH}")

Capa Gold escrita en: /opt/UNIDAD1/gold/estaciones_utilizacion


In [12]:
df_gold = spark.read.parquet(GOLD_PATH)

n_escrito = df_etiquetado.count()
n_leido = df_gold.count()
print(f"Filas escritas : {n_escrito:,}")
print(f"Filas leidas   : {n_leido:,}")
print(f"Conteo reconciliado: {n_escrito == n_leido}")

print("\nPlan fisico al filtrar por es_alta_demanda = 1 (debe mostrar PartitionFilters):")
df_gold.filter(col("es_alta_demanda") == 1).explain(True)

Filas escritas : 70,388
Filas leidas   : 70,388
Conteo reconciliado: True

Plan fisico al filtrar por es_alta_demanda = 1 (debe mostrar PartitionFilters):
== Parsed Logical Plan ==
'Filter '`=`('es_alta_demanda, 1)
+- Relation [station_id#9686,station_name#9687,trip_date#9688,day_of_week#9689,is_weekend#9690,viajes_totales_dia#9691L,member_ratio#9692,electric_ratio#9693,es_alta_demanda#9694] parquet

== Analyzed Logical Plan ==
station_id: string, station_name: string, trip_date: date, day_of_week: int, is_weekend: int, viajes_totales_dia: bigint, member_ratio: double, electric_ratio: double, es_alta_demanda: int
Filter (es_alta_demanda#9694 = 1)
+- Relation [station_id#9686,station_name#9687,trip_date#9688,day_of_week#9689,is_weekend#9690,viajes_totales_dia#9691L,member_ratio#9692,electric_ratio#9693,es_alta_demanda#9694] parquet

== Optimized Logical Plan ==
Filter (isnotnull(es_alta_demanda#9694) AND (es_alta_demanda#9694 = 1))
+- Relation [station_id#9686,station_name#9687,trip_dat

In [13]:
GOLD_RANKING_PATH = "/opt/UNIDAD1/gold/estaciones_ranking"
df_ranking.write.mode("overwrite").parquet(GOLD_RANKING_PATH)
print(f"Ranking de estaciones persistido en: {GOLD_RANKING_PATH}")

Ranking de estaciones persistido en: /opt/UNIDAD1/gold/estaciones_ranking


## 12. Componente ML distribuido: clasificación de alta demanda (S4)

* `StringIndexer` + `OneHotEncoder` para `day_of_week` (variable categórica cíclica, no ordinal).
* `VectorAssembler` combina `day_of_week_ohe`, `is_weekend`, `member_ratio` y `electric_ratio` en `features`.
* Split **temporal** (no aleatorio): entrenamos con los primeros días del rango y evaluamos sobre los
  días finales — evita que el modelo "vea" el futuro y refleja el uso real (predecir demanda de días
  que aún no ocurrieron).

In [14]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

df_train_lbl = df_etiquetado.filter(col("trip_date") < lit(corte))
df_test_lbl = df_etiquetado.filter(col("trip_date") >= lit(corte))

print(f"Train: {df_train_lbl.count():,} filas | Test: {df_test_lbl.count():,} filas")

indexer = StringIndexer(inputCol="day_of_week", outputCol="day_of_week_idx", handleInvalid="keep")
encoder = OneHotEncoder(inputCols=["day_of_week_idx"], outputCols=["day_of_week_ohe"])
assembler = VectorAssembler(
    inputCols=["day_of_week_ohe", "is_weekend", "member_ratio", "electric_ratio"],
    outputCol="features",
)

prep_pipeline = Pipeline(stages=[indexer, encoder, assembler])
prep_model = prep_pipeline.fit(df_train_lbl)

train = prep_model.transform(df_train_lbl)
test = prep_model.transform(df_test_lbl)

Train: 52,285 filas | Test: 18,103 filas


## 13. Entrenamiento y comparación de configuraciones (S4)

Se entrenan **tres configuraciones**: regresión logística sin regularización, regresión logística
con regularización L2 (`regParam=0.1`) y un `RandomForestClassifier`; todas se evalúan sobre el mismo
conjunto de prueba con **tres métricas**: `areaUnderROC`, `f1` y `accuracy`.

In [15]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

label_col = "es_alta_demanda"

lr_base = LogisticRegression(featuresCol="features", labelCol=label_col, regParam=0.0)
lr_reg = LogisticRegression(featuresCol="features", labelCol=label_col, regParam=0.1, elasticNetParam=0.0)
rf = RandomForestClassifier(featuresCol="features", labelCol=label_col, numTrees=50, maxDepth=6, seed=42)

modelos = {
    "LogisticRegression (sin regularizacion)": lr_base,
    "LogisticRegression (L2, regParam=0.1)": lr_reg,
    "RandomForestClassifier (50 arboles)": rf,
}

auc_eval = BinaryClassificationEvaluator(labelCol=label_col, metricName="areaUnderROC")
f1_eval = MulticlassClassificationEvaluator(labelCol=label_col, metricName="f1")
acc_eval = MulticlassClassificationEvaluator(labelCol=label_col, metricName="accuracy")

resultados = []
modelos_entrenados = {}

for nombre, estimador in modelos.items():
    modelo = estimador.fit(train)
    preds = modelo.transform(test)
    auc = auc_eval.evaluate(preds)
    f1 = f1_eval.evaluate(preds)
    acc = acc_eval.evaluate(preds)
    resultados.append((nombre, round(auc, 4), round(f1, 4), round(acc, 4)))
    modelos_entrenados[nombre] = modelo

print(f"{'Modelo':42s} {'AUC':>8s} {'F1':>8s} {'Accuracy':>10s}")
print("-" * 72)
for nombre, auc, f1, acc in resultados:
    print(f"{nombre:42s} {auc:8.4f} {f1:8.4f} {acc:10.4f}")

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


Modelo                                          AUC       F1   Accuracy
------------------------------------------------------------------------
LogisticRegression (sin regularizacion)      0.6386   0.5894     0.7026
LogisticRegression (L2, regParam=0.1)        0.6352   0.5932     0.7109
RandomForestClassifier (50 arboles)          0.7861   0.5947     0.7139


## 14. Selección y persistencia del modelo ganador (S4)

Se elige el modelo con mayor `areaUnderROC` sobre el conjunto de prueba y se guarda **solo ese**
modelo (no el primero entrenado).

In [16]:
mejor_nombre, mejor_auc, mejor_f1, mejor_acc = max(resultados, key=lambda r: r[1])
mejor_modelo = modelos_entrenados[mejor_nombre]

print(f"Modelo ganador: {mejor_nombre}")
print(f"  AUC      : {mejor_auc}")
print(f"  F1       : {mejor_f1}")
print(f"  Accuracy : {mejor_acc}")

MODEL_PATH = "/opt/UNIDAD1/gold/modelo_estaciones_alta_demanda"
mejor_modelo.write().overwrite().save(MODEL_PATH)
print(f"\nModelo ganador persistido en: {MODEL_PATH}")

Modelo ganador: RandomForestClassifier (50 arboles)
  AUC      : 0.7861
  F1       : 0.5947
  Accuracy : 0.7139



Modelo ganador persistido en: /opt/UNIDAD1/gold/modelo_estaciones_alta_demanda


## 15. Verificación de reproducibilidad del modelo guardado

In [17]:
from pyspark.ml.classification import LogisticRegressionModel, RandomForestClassificationModel

try:
    modelo_cargado = LogisticRegressionModel.load(MODEL_PATH)
except Exception:
    modelo_cargado = RandomForestClassificationModel.load(MODEL_PATH)

preds_verif = modelo_cargado.transform(test)
auc_verif = auc_eval.evaluate(preds_verif)
print(f"AUC del modelo recargado desde disco: {auc_verif:.4f} (esperado: {mejor_auc})")
print(f"Coincide con la evaluacion original: {round(auc_verif, 4) == mejor_auc}")

AUC del modelo recargado desde disco: 0.7861 (esperado: 0.7861)
Coincide con la evaluacion original: True


## 16. Síntesis para la sustentación

1. **Esquema explícito no negociable:** los IDs de estación se leen como `String`; inferirlos como
   `double` corrompe silenciosamente cualquier agrupación por estación.
2. **RDD y DataFrame conciliados:** el conteo de viajes por estación se calculó por dos caminos
   (`groupBy().agg()` y `map/reduceByKey` sobre RDD) y ambos coinciden exactamente.
3. **Calidad documentada, no solo mencionada:** deduplicación por `ride_id` y descarte de nulos en
   campos de estación, con conteos de antes/después en cada paso.
4. **Sin fuga de información:** el umbral de "alta demanda" y el split train/test son temporales —
   el modelo nunca ve datos del futuro al momento de entrenar ni al definir el umbral de la etiqueta.
5. **Comparación real antes de guardar:** tres configuraciones (dos algoritmos, con y sin
   regularización) evaluadas con las mismas tres métricas; se persiste únicamente la ganadora.
6. **Puente hacia Unidad II:** esta misma pregunta (¿qué estaciones tendrán alta demanda?) se
   responde en streaming sobre `citibike-rides` con Structured Streaming, usando el histórico de
   esta capa Gold para entrenar el modelo de ventana temporal.